# V1-S05 Embedding Bake-off

Pick the best embedder for the thematic backbone via MeSH pseudo-labels on a 500-abstract sample.

Per plan D6 / D12: stratified MeSH sample → three embedders (mpnet, bge-large, nomic) → metrics (kNN@10 precision, intra/inter cosine separation, runtime/1k) → decision rule with stop-condition guard.

## 1. Setup

In [1]:
from __future__ import annotations

import os
import time
from pathlib import Path

# Mac CPU is the bake-off target (per locked decision #2). Force CPU + thread
# limits to avoid Apple Silicon MPS segfaults during sentence-transformer encode.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import duckdb
import numpy as np
import pandas as pd
import torch

torch.set_num_threads(2)

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

DUCKDB_PATH = REPO_ROOT / "data/v1/papers.duckdb"
SAMPLE_PATH = REPO_ROOT / "data/v1/embedding_bakeoff_sample.parquet"
SAMPLE_PATH.parent.mkdir(parents=True, exist_ok=True)

print("DuckDB:", DUCKDB_PATH, DUCKDB_PATH.exists())
con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
print("Tables:", con.execute("SHOW TABLES").fetchall())

DuckDB: /Users/samersalman/Desktop/SciField/data/v1/papers.duckdb True
Tables: [('authorships',), ('citation_intents',), ('enrichment_failed',), ('institutions',), ('journals',), ('mesh',), ('openalex_works',), ('paper_institutions',), ('papers',), ('references_out',), ('semantic_scholar',)]


In [2]:
# Abstract-bearing papers and MeSH coverage.
# papers.duckdb stores MeSH as a STRUCT column `heading` on the `mesh` view; the
# topical descriptor is `heading.descriptor`. We exclude "check-tag" descriptors
# (Humans, Female, Male, Adult, etc.) by filtering out any descriptor that
# appears on > MAX_DESCRIPTOR_PCT of abstract-bearing papers — these carry no
# semantic-clustering signal.

n_abstract = con.execute(
    "SELECT COUNT(DISTINCT pmid) FROM papers "
    "WHERE abstract IS NOT NULL AND length(abstract) > 50"
).fetchone()[0]
print(f"abstract-bearing papers: {n_abstract:,}")

MAX_DESCRIPTOR_PCT = 25  # drop any descriptor on > 25% of papers (check tags / demographics)

mesh_counts = con.execute(f"""
    WITH md AS (SELECT pmid, heading.descriptor AS mesh FROM mesh),
         abs_papers AS (
             SELECT DISTINCT pmid FROM papers
             WHERE abstract IS NOT NULL AND length(abstract) > 50
         )
    SELECT md.mesh, COUNT(DISTINCT md.pmid) AS n
    FROM md JOIN abs_papers USING (pmid)
    GROUP BY md.mesh
    HAVING 100.0 * COUNT(DISTINCT md.pmid) / {n_abstract} <= {MAX_DESCRIPTOR_PCT}
    ORDER BY n DESC
    LIMIT 50
""").fetchdf()
print(mesh_counts.head(20))

abstract-bearing papers: 89,230
                               mesh      n
0                 Aged, 80 and over  15756
1                        Adolescent  12866
2                 Follow-Up Studies  12229
3       Postoperative Complications  12085
4               Prospective Studies  11345
5    Arthroplasty, Replacement, Hip   9030
6                      Risk Factors   8630
7                       Reoperation   8534
8   Arthroplasty, Replacement, Knee   8112
9                       Radiography   7773
10                      Young Adult   7684
11                     Time Factors   7444
12                          Animals   6575
13                    United States   6140
14                            Child   5950
15                      Arthroscopy   5755
16       Range of Motion, Articular   5751
17                       Knee Joint   5456
18                 Lumbar Vertebrae   5246
19                   Hip Prosthesis   4811


## 2. Sample construction (500 papers, stratified by MeSH)

Stratified random sample of 500 abstract-bearing papers across the top-K most frequent MeSH descriptors (K chosen so each cell has ≥ 20 papers). Single-label assignment per paper (rarest descriptor wins ordering).

In [ ]:
SAMPLE_SIZE = 500
MIN_PER_CELL = 20

# Iterate K downward until each of the top-K MeSH descriptors yields ≥ MIN_PER_CELL distinct
# abstract-bearing papers under a single-label assignment. The label assigned to each paper
# is its *rarest* (most-informative) descriptor among the top-K — this biases labels toward
# topical terms (e.g. "Arthroscopy") rather than demographic check-tags (e.g. "Adolescent").
K = 25
df = None
while K >= 5:
    top_df = mesh_counts.head(K)
    top = top_df["mesh"].tolist()
    # rarity rank inside the top-K: smaller n = rarer = more informative.
    rarity = {row.mesh: int(row.n) for row in top_df.itertuples()}
    placeholders = ",".join([f"'{m.replace(chr(39), chr(39)*2)}'" for m in top])

    df = con.execute(f"""
        WITH paper_mesh AS (
            SELECT p.pmid, p.title, p.abstract, m.heading.descriptor AS mesh
            FROM papers p
            JOIN mesh m USING (pmid)
            WHERE p.abstract IS NOT NULL AND length(p.abstract) > 50
              AND m.heading.descriptor IN ({placeholders})
        )
        SELECT DISTINCT pmid, title, abstract, mesh FROM paper_mesh
    """).fetchdf()
    # Rarest-first single-label per paper.
    df = df.assign(_rarity=df["mesh"].map(rarity))
    df = (
        df.sort_values(["pmid", "_rarity", "mesh"])
        .drop_duplicates("pmid", keep="first")
        .drop(columns="_rarity")
    )

    per_label = df["mesh"].value_counts()
    if (per_label >= MIN_PER_CELL).all():
        print(f"K={K} works; min cell = {per_label.min()}, max cell = {per_label.max()}")
        break
    K -= 1
else:
    raise RuntimeError(f"could not find K with ≥{MIN_PER_CELL} per cell")

# Stratified sample: ceil(SAMPLE_SIZE/K) from each label, truncated to SAMPLE_SIZE.
rng = np.random.default_rng(seed=42)
per_k = int(np.ceil(SAMPLE_SIZE / K))
groups = []
for _label, sub in df.groupby("mesh"):
    n = min(per_k, len(sub))
    groups.append(sub.sample(n=n, random_state=rng.integers(0, 1_000_000)))
sample = pd.concat(groups).sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print("sample shape:", sample.shape)
print(sample["mesh"].value_counts().head(15))

sample.to_parquet(SAMPLE_PATH, index=False)
print("wrote", SAMPLE_PATH)

In [4]:
from scifield.repro import record_run

record_run(
    artifact_path=SAMPLE_PATH,
    inputs={"papers_duckdb": DUCKDB_PATH},
    config={"sample_size": SAMPLE_SIZE, "K": int(K), "min_per_cell": MIN_PER_CELL, "seed": 42},
)
print("sidecar:", SAMPLE_PATH.with_suffix(SAMPLE_PATH.suffix + ".run.json"))

sidecar: /Users/samersalman/Desktop/SciField/data/v1/embedding_bakeoff_sample.parquet.run.json


## 3. Run 3 embedders on the 500

Runs locally on Mac CPU (≈10–20 min total). All embedders L2-normalise their outputs internally; wall-clock per 1k recorded.

In [ ]:
from scifield.thematic.embed import make_embedder

sample = pd.read_parquet(SAMPLE_PATH)
texts = [
    (t or "") + ". " + (a or "") for t, a in zip(sample["title"], sample["abstract"], strict=False)
]
labels = sample["mesh"].to_numpy()

EMBEDDERS = ["mpnet", "bge-large-en-v1.5", "nomic-embed-text-v1"]
results = {}  # name -> {"vectors": ndarray, "runtime_s": float}
failed = {}  # name -> exception string

for name in EMBEDDERS:
    print(f"--- {name} ---")
    try:
        # CPU-forced per setup-cell rationale (avoid MPS segfaults).
        emb = make_embedder(name, device="cpu")
        t0 = time.perf_counter()
        vecs = emb.encode(texts, batch_size=16)
        runtime = time.perf_counter() - t0
        per_1k = runtime / len(texts) * 1000
        print(
            f"  shape={vecs.shape}, dtype={vecs.dtype}, "
            f"runtime={runtime:.1f}s, per_1k={per_1k:.1f}s"
        )
        results[name] = {"vectors": vecs, "runtime_s": runtime}
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        print(f"  FAILED — {msg}")
        failed[name] = msg

if not results:
    raise RuntimeError(f"All embedders failed: {failed}")
print(f"\nran: {list(results)}; failed: {list(failed)}")

## 4. Metrics table

Per D6: intra/inter cosine separation (higher better), kNN@10 precision (higher better), runtime per 1k (lower better).

In [6]:
def intra_inter_separation(vecs: np.ndarray, labels: np.ndarray) -> float:
    # Cosine = inner product on L2-normalised vectors.
    sim = vecs @ vecs.T
    same = labels[:, None] == labels[None, :]
    np.fill_diagonal(same, False)  # exclude self
    intra = sim[same].mean()
    inter = sim[~same].mean()
    return float(intra - inter)


def knn_precision_at_10(vecs: np.ndarray, labels: np.ndarray, k: int = 10) -> float:
    sim = vecs @ vecs.T
    np.fill_diagonal(sim, -np.inf)
    topk = np.argsort(-sim, axis=1)[:, :k]
    same = labels[topk] == labels[:, None]
    return float(same.mean())


rows = []
for name, r in results.items():
    vecs = r["vectors"]
    rows.append(
        {
            "model": name,
            "knn@10_precision": knn_precision_at_10(vecs, labels),
            "intra_minus_inter_cos": intra_inter_separation(vecs, labels),
            "runtime_per_1k_s": r["runtime_s"] / len(texts) * 1000,
            "dim": vecs.shape[1],
        }
    )
metrics = pd.DataFrame(rows).set_index("model")
metrics

,knn@10_precision,intra_minus_inter_cos,runtime_per_1k_s,dim
model,,,,
mpnet,0.1494,0.106794,80.916704,768
bge-large-en-v1.5,0.1608,0.035338,245.058973,1024
nomic-embed-text-v1,0.1626,0.033447,130.778118,768


## 5. Recommendation (with stop-condition guard)

D6 decision rule: highest kNN@10 wins; tiebreak by intra/inter separation. **Stop-condition guard:** if the winner does not beat `all-mpnet-base-v2` by ≥ 0.03 on kNN@10 precision, default to mpnet.

In [7]:
# D6 decision rule + stop-condition guard, robust to any failed embedders.
mpnet_p = metrics.loc["mpnet", "knn@10_precision"] if "mpnet" in metrics.index else None
best = metrics.sort_values("knn@10_precision", ascending=False)
top_name = best.index[0]
top_p = best.iloc[0]["knn@10_precision"]

if top_name == "mpnet":
    winner = "mpnet"
    reason = f"mpnet wins outright (knn@10={top_p:.3f})"
elif mpnet_p is None:
    # mpnet failed to run — pick the available winner by D6 rule, document deviation.
    winner = top_name
    reason = (
        f"mpnet failed to run; picking top candidate {top_name} by D6 rule "
        f"(knn@10={top_p:.3f}). Stop-condition guard could not be applied."
    )
elif top_p - mpnet_p >= 0.03:
    winner = top_name
    reason = f"{top_name} beats mpnet by Δ={top_p-mpnet_p:.3f} (≥ 0.03 stop-condition threshold)"
else:
    winner = "mpnet"
    reason = (
        f"stop-condition guard fired: best candidate {top_name} only beats mpnet "
        f"by Δ={top_p-mpnet_p:.3f} (< 0.03). Defaulting to mpnet."
    )

print(f"WINNER: {winner}")
print(f"Reason: {reason}")
if failed:
    print("\nDeviation: the following embedders failed and were excluded from the bake-off:")
    for k, v in failed.items():
        print(f"  - {k}: {v}")
print()
print(metrics.round(4))

WINNER: mpnet
Reason: stop-condition guard fired: best candidate nomic-embed-text-v1 only beats mpnet by Δ=0.013 (< 0.03). Defaulting to mpnet.

                     knn@10_precision  intra_minus_inter_cos  \
model                                                          
mpnet                          0.1494                 0.1068   
bge-large-en-v1.5              0.1608                 0.0353   
nomic-embed-text-v1            0.1626                 0.0334   

                     runtime_per_1k_s   dim  
model                                        
mpnet                         80.9167   768  
bge-large-en-v1.5            245.0590  1024  
nomic-embed-text-v1          130.7781   768  


**Next step:** update `conf/thematic/embed.yaml` per the winner above.
Pin `model.revision` to a commit SHA (look up the model on HuggingFace and copy the SHA of the `main` branch HEAD at this run's time).
Then run the Brev embedding via `bash scripts/brev_embed.sh`.

## 6. FAISS spot-check

Run **after** `data/v1/faiss.index` exists locally (built via `scifield faiss-build`). Inspect 10 hand-chosen PMIDs and their top-5 nearest neighbours. Visually plausible neighbour clusters validates the index.

In [8]:
from scifield.thematic.faiss_index import read_index, read_pmid_map

FAISS_PATH = REPO_ROOT / "data/v1/faiss.index"
PMID_MAP_PATH = REPO_ROOT / "data/v1/faiss_pmid_map.parquet"

# 10 hand-chosen PMIDs — one mid-period paper from each of the corpus's 10 main journals,
# chosen for topical diversity (arthroplasty, spine, COR&R, arthroscopy, general/vascular/
# colorectal/upper-GI/endocrine surgery, JBJS). Used for the V1-S05 §8 spot-check.
QUERY_PMIDS = [
    28579444,  # J Arthroplasty — posterior tibial slope analysis
    22695276,  # Spine — nucleus-annulus integration
    21384209,  # CORR — amputation disparities
    29413187,  # Arthroscopy — scapula tilt editorial
    29108705,  # Surgery — metabolic syndrome + postop complications
    28338510,  # Ann Surg — laparoscopic lavage vs resection in diverticulitis
    24430426,  # JBJS — calcaneal stress fracture post-TKA/THA
    24474187,  # Br J Surg — co-morbidity + QOL after oesophagectomy
    25260683,  # JACS — methylated CpG sites
    18427019,  # Arch Surg — hyperparathyroidism surgery, image-negative
]

if not FAISS_PATH.exists():
    print(f"{FAISS_PATH} does not exist yet — run scifield embed + scifield faiss-build first.")
    index = None
else:
    index = read_index(FAISS_PATH, ef_search=64)
    row_ids, pmids = read_pmid_map(PMID_MAP_PATH)
    pmid_to_row = {int(p): int(r) for r, p in zip(row_ids, pmids, strict=False)}
    print(f"index.ntotal = {index.ntotal}")
    print(
        f"all {len(QUERY_PMIDS)} query PMIDs present in index:",
        all(p in pmid_to_row for p in QUERY_PMIDS),
    )

index.ntotal = 99938
all 10 query PMIDs present in index: True


In [ ]:
if index is not None and QUERY_PMIDS:
    con2 = duckdb.connect(str(DUCKDB_PATH), read_only=True)
    # papers.pmid is VARCHAR; cast both sides to BIGINT for clean joins.
    placeholders = ",".join(str(p) for p in QUERY_PMIDS)
    query_titles = con2.execute(
        f"SELECT CAST(pmid AS BIGINT) AS pmid, title, journal "
        f"FROM papers WHERE CAST(pmid AS BIGINT) IN ({placeholders})"
    ).fetchdf()

    # Reconstruct query vectors from the index and search top-6 (rank-0 = self).
    rows = [pmid_to_row[p] for p in QUERY_PMIDS]
    q_vecs = np.vstack([index.reconstruct(r) for r in rows]).astype(np.float32)
    sims_mat, idx_mat = index.search(q_vecs, k=6)

    nn_records = []
    for qp, neighbours, sims in zip(QUERY_PMIDS, idx_mat, sims_mat, strict=False):
        for rank, (n_row, sim) in enumerate(zip(neighbours, sims, strict=False)):
            n_pmid = int(pmids[n_row])
            if n_pmid == qp:
                continue
            nn_records.append(
                {"query_pmid": qp, "rank": rank, "nn_pmid": n_pmid, "sim": round(float(sim), 3)}
            )

    nn_df = pd.DataFrame(nn_records)
    all_nn = nn_df["nn_pmid"].unique().tolist()
    nn_placeholders = ",".join(str(p) for p in all_nn)
    nn_titles = con2.execute(
        f"SELECT CAST(pmid AS BIGINT) AS pmid, title, journal "
        f"FROM papers WHERE CAST(pmid AS BIGINT) IN ({nn_placeholders})"
    ).fetchdf()
    nn_full = nn_df.merge(nn_titles, left_on="nn_pmid", right_on="pmid")

    print("=== top-5 neighbours per query ===")
    for qp in QUERY_PMIDS:
        qrow = query_titles[query_titles["pmid"] == qp].iloc[0]
        print(f"\nQ {qp:>9}  [{qrow['journal'][:32]:32s}]  {qrow['title'][:80]}")
        sub = nn_full[nn_full["query_pmid"] == qp].sort_values("rank").head(5)
        for _, r in sub.iterrows():
            print(
                f"  r={int(r['rank'])} sim={r['sim']:.3f}  "
                f"{int(r['nn_pmid']):>9}  [{r['journal'][:25]:25s}]  {r['title'][:60]}"
            )